# 2. From pixels to landmarks

**Computer vision** is the field of teaching computers to understand images/video.

A raw video frame is just millions of pixel numbers — the model has no idea
where the hands are. **MediaPipe** is a Google library that finds specific
body points (landmarks) in an image, e.g. fingertip positions, wrist, elbows.

A **landmark** = one (x, y, z) coordinate for one specific point on the body.

For sign language:
- **Hand landmarks** (21 points per hand) capture finger shapes.
- **Pose landmarks** (33 points) capture arm/shoulder movement — important
  because many ISL signs use arm position and motion, not just hand shape.

```text
ISL Video Frame
      ↓
  MediaPipe
      ↓
Hand landmarks + Pose landmarks
      ↓
Numbers (features) the model can actually learn from
```

**Tenglish:** "Model ki video direct ga ardham kaadu. Manam video ni
numbers/features ga convert chesi model ki istam — aa numbers ne landmarks
antaru."

**Note on the MediaPipe API:** this notebook uses MediaPipe's current
**Tasks API** (`HandLandmarker`, `PoseLandmarker`). The older `mp.solutions`
API used in many tutorials has been removed from recent MediaPipe versions.

## 0. Setup (Colab or local)

In [ ]:
import os

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/ISL_Sign2Sub"
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = os.path.abspath("..")
    ON_COLAB = False

VIDEOS_DIR = os.path.join(PROJECT_DIR, "data", "videos")
LANDMARKS_DIR = os.path.join(PROJECT_DIR, "data", "landmarks")
os.makedirs(LANDMARKS_DIR, exist_ok=True)

print("Running on Colab:", ON_COLAB)
print("Landmarks will be saved to:", LANDMARKS_DIR)

## 1. Install MediaPipe and download the two detector models

In [ ]:
!pip install -q mediapipe
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode

hand_landmarker = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
        running_mode=VisionRunningMode.IMAGE,
        num_hands=2,
    )
)

pose_landmarker = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="pose_landmarker.task"),
        running_mode=VisionRunningMode.IMAGE,
    )
)

print("MediaPipe Tasks ready.")

## 2. Load frames from one video

In [ ]:
def read_frames(path):
    cap = cv2.VideoCapture(path)
    out = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        out.append(fr)
    cap.release()
    return out

video_path = None
for root, dirs, files in os.walk(VIDEOS_DIR):
    for f in files:
        if f.lower().endswith((".mp4", ".avi", ".mov")):
            video_path = os.path.join(root, f)
            break
    if video_path:
        break

print("Using video:", video_path)
frames = read_frames(video_path)
print("Loaded frames:", len(frames))

## 3. Run MediaPipe on one frame and draw the landmarks

**Tenglish:** "Landmark ante image lo oka point. MediaPipe ichhe x, y values
0 nunchi 1 madhya untayi, so vaatini width, height tho multiply chesthe
pixel position vastundi."

In [ ]:
def frame_to_mp_image(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

def draw_landmarks(frame_bgr, hand_result, pose_result):
    img = frame_bgr.copy()
    h, w = img.shape[:2]
    for hand in hand_result.hand_landmarks:
        for lm in hand:
            cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 5, (0, 255, 0), -1)
    for pose in pose_result.pose_landmarks:
        for lm in pose:
            cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 5, (0, 0, 255), -1)
    return img

idx = len(frames) // 2
mp_image = frame_to_mp_image(frames[idx])
hand_result = hand_landmarker.detect(mp_image)
pose_result = pose_landmarker.detect(mp_image)

print("Hands detected:", len(hand_result.hand_landmarks))
print("Landmarks per hand:", [len(h) for h in hand_result.hand_landmarks])
print("Pose landmarks:", len(pose_result.pose_landmarks[0]) if pose_result.pose_landmarks else 0)

annotated = draw_landmarks(frames[idx], hand_result, pose_result)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(frames[idx], cv2.COLOR_BGR2RGB))
axes[0].set_title("Original frame")
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title("With landmarks")
for ax in axes:
    ax.axis("off")
plt.show()

## 4. Turn one frame's landmarks into 225 numbers

```text
Left hand:  21 points x (x, y, z) = 63 numbers
Right hand: 21 points x (x, y, z) = 63 numbers
Body pose:  33 points x (x, y, z) = 99 numbers
                                    ---------
Total per frame                     225 numbers
```

A missing hand is filled with zeros. Each hand goes in a fixed slot (left
vs right) using MediaPipe's own handedness label.

In [ ]:
def extract_frame_features(frame_bgr):
    """Returns 225 numbers for one frame: left hand + right hand + pose."""
    mp_image = frame_to_mp_image(frame_bgr)
    hand_result = hand_landmarker.detect(mp_image)
    pose_result = pose_landmarker.detect(mp_image)

    left = np.zeros(63, dtype=np.float32)
    right = np.zeros(63, dtype=np.float32)
    pose = np.zeros(99, dtype=np.float32)

    for landmarks, handedness in zip(hand_result.hand_landmarks, hand_result.handedness):
        coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).flatten()
        if handedness[0].category_name == "Left":
            left = coords
        else:
            right = coords

    if pose_result.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z] for lm in pose_result.pose_landmarks[0]],
                        dtype=np.float32).flatten()

    return np.concatenate([left, right, pose])

features = extract_frame_features(frames[len(frames) // 2])
print("Shape:", features.shape)
print("Left hand missing?", np.all(features[:63] == 0))
print("Right hand missing?", np.all(features[63:126] == 0))

## 5. Do it for every frame and stack the results

```text
Frame 1  -> 225 numbers
Frame 2  -> 225 numbers
...
Frame N  -> 225 numbers
         ↓
Sequence of shape (N, 225)
```

In [ ]:
import time

def extract_video_features(frames):
    return np.stack([extract_frame_features(f) for f in frames])

start = time.time()
sequence = extract_video_features(frames)
print("Time taken: %.1f seconds" % (time.time() - start))
print("Sequence shape:", sequence.shape)

plt.figure(figsize=(10, 4))
plt.plot(sequence[:, 0], label="Left wrist x")
plt.plot(sequence[:, 63], label="Right wrist x")
plt.xlabel("Frame number"); plt.ylabel("x position")
plt.title("Wrist movement over time"); plt.legend()
plt.show()

## 6. Save one sequence to a `.npy` file

Filenames combine sign name + video name (`HELLO__MVI_0029.npy`), since
different sign folders reuse the same video filenames.

In [ ]:
label = os.path.basename(os.path.dirname(video_path))
video_name = os.path.splitext(os.path.basename(video_path))[0]
save_path = os.path.join(LANDMARKS_DIR, f"{label}__{video_name}.npy")

np.save(save_path, sequence)
print("Saved to:", save_path)

loaded = np.load(save_path)
print("Same as original?", np.array_equal(sequence, loaded))

## 7. Run this on ALL your videos

Skips videos already saved (resumable if Colab disconnects), and catches
errors per video.

In [ ]:
all_videos = []
for root, dirs, files in os.walk(VIDEOS_DIR):
    for f in sorted(files):
        if f.lower().endswith((".mp4", ".avi", ".mov")):
            all_videos.append(os.path.join(root, f))
print("Total videos found:", len(all_videos))

done, skipped, failed = 0, 0, []
start = time.time()

for i, vpath in enumerate(all_videos, 1):
    lbl = os.path.basename(os.path.dirname(vpath))
    name = os.path.splitext(os.path.basename(vpath))[0]
    out_path = os.path.join(LANDMARKS_DIR, f"{lbl}__{name}.npy")
    if os.path.exists(out_path):
        skipped += 1
        continue
    try:
        vid_frames = read_frames(vpath)
        seq = np.stack([extract_frame_features(f) for f in vid_frames])
        np.save(out_path, seq)
        done += 1
    except Exception as e:
        failed.append((vpath, str(e)))
    if i % 10 == 0:
        print(f"{i}/{len(all_videos)} processed | {(time.time()-start)/60:.1f} min elapsed")

print("\nFinished.")
print("Newly saved:", done, "| Skipped:", skipped, "| Failed:", len(failed))
for p, err in failed:
    print("FAILED:", p, "->", err)

## 8. Check the results

In [ ]:
from collections import Counter

files = [f for f in os.listdir(LANDMARKS_DIR) if f.endswith(".npy")]
print("Total .npy files:", len(files))
print(Counter(f.split("__")[0] for f in files))

## 9. (Optional) Add more sign categories from the full INCLUDE dataset

Downloads+extracts one category at a time on Colab's temporary disk, saves
only `.npy` landmark files to Drive, then deletes the videos — so Drive
never fills up with raw video.

Needs `files.txt`. If missing in this session:
```python
!curl -s https://zenodo.org/api/records/4010759 -o metadata.json
!apt-get install -y jq
!cat metadata.json | jq -r '.files[] | .links.self + " " + .key' > files.txt
```

In [ ]:
import subprocess, shutil, re

def sanitize(folder_name):
    name = re.sub(r"^\d+\.\s*", "", folder_name)
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").upper()

def process_category(category, scratch="/content/work"):
    entries = [l.split() for l in open("files.txt") if l.split()[1].lower().startswith(category.lower())]
    shutil.rmtree(scratch, ignore_errors=True); os.makedirs(scratch)

    for url, key in entries:
        subprocess.run(["curl", "-s", "-C", "-", "-o", f"{scratch}/{key}", url], check=True)
        subprocess.run(["unzip", "-q", "-o", f"{scratch}/{key}", "-d", scratch], check=True)
        os.remove(f"{scratch}/{key}")

    videos = [os.path.join(r, f) for r, _, fs in os.walk(scratch) for f in sorted(fs)
              if f.lower().endswith((".mp4", ".avi", ".mov"))]
    print(f"{category}: {len(videos)} videos")

    dn, sk, t0 = 0, 0, time.time()
    for v in videos:
        lbl = sanitize(os.path.basename(os.path.dirname(v)))
        out = os.path.join(LANDMARKS_DIR, f"{lbl}__{os.path.splitext(os.path.basename(v))[0]}.npy")
        if os.path.exists(out):
            sk += 1; continue
        try:
            fr = read_frames(v)
            np.save(out, np.stack([extract_frame_features(f) for f in fr]))
            dn += 1
        except Exception as e:
            print("FAILED:", v, e)
        if (dn + sk) % 20 == 0:
            print(f"  {dn + sk}/{len(videos)} | {(time.time()-t0)/60:.1f} min")

    shutil.rmtree(scratch, ignore_errors=True)
    print(f"Finished {category}: saved {dn}, skipped {sk}")

# To use:
# keys = [l.split()[1] for l in open("files.txt")]
# categories = sorted(set(re.sub(r"_\d+of\d+\.zip$", "", k) for k in keys))
# print(categories)
# for cat in categories:
#     if cat.lower() == "greetings":
#         continue
#     process_category(cat)

print("process_category() ready. Uncomment the loop above to run it on more categories.")